In [2]:
import pandas as pd
import numpy as np

## 加载文件

In [12]:
import pandas as pd

# 读取 CSV 文件并保存为 df
file_path = "./pre_2023-2024_with_comp_test.csv"

# 确保文件存在后读取
try:
    df = pd.read_csv(file_path)
    print("✅ 文件成功加载！")
    print(f"📊 数据维度: {df.shape}")
    print("📌 前5行数据:")
    display(df.head())
except FileNotFoundError:
    print(f"❌ 错误: 文件 {file_path} 未找到，请检查文件路径！")


✅ 文件成功加载！
📊 数据维度: (282755, 20)
📌 前5行数据:


,flt_no,cap,aircraft,legs,leg_no,duration,pax,a,b,c,unit_price,competitor_price,year,month,day,weekday,hour,minute,from,to
0,7558,94,190,1,1,1.33,45,AAT,URC,NaN,481.555556,-119.744444,2024,7,1,0,12,30,AAT,URC
1,5248,162,322,1,1,2.10,143,AKA,HGH,NaN,902.167832,0.000000,2024,7,1,0,10,30,AKA,HGH
2,3524,169,738,1,1,4.52,154,AKU,CGO,NaN,1540.357143,283.690476,2024,7,1,0,14,15,AKU,CGO
3,6352,167,320,1,1,4.28,167,AKU,CGO,NaN,1345.449102,88.782435,2024,7,1,0,21,30,AKU,CGO
4,7516,166,320,1,1,1.33,181,AKU,URC,NaN,593.480663,-54.049640,2024,7,1,0,23,15,AKU,URC


## 计算票价信息

## 保存原始文件

## 保留甩飞行(legs=3)

In [13]:
# 统计legs的值计数
legs_count = df['legs'].value_counts().sort_index()

# 显示统计结果
print("Legs分布情况：")
print(legs_count)

# 显示百分比
legs_percentage = df['legs'].value_counts(normalize=True).sort_index() * 100
print("\nLegs分布百分比：")
print(legs_percentage.round(2), "%")

Legs分布情况：
legs
1    170447
3    112308
Name: count, dtype: int64

Legs分布百分比：
legs
1    60.28
3    39.72
Name: proportion, dtype: float64 %


In [14]:
df = df[df['legs'] == 3]

In [15]:
# 统计legs的值计数
legs_count = df['legs'].value_counts().sort_index()

# 显示统计结果
print("Legs分布情况：")
print(legs_count)

# 显示百分比
legs_percentage = df['legs'].value_counts(normalize=True).sort_index() * 100
print("\nLegs分布百分比：")
print(legs_percentage.round(2), "%")

Legs分布情况：
legs
3    112308
Name: count, dtype: int64

Legs分布百分比：
legs
3    100.0
Name: proportion, dtype: float64 %


## 合并航线

In [16]:
import pandas as pd
import numpy as np
from tqdm import tqdm  # 导入tqdm库用于显示进度条

# 假设数据已经加载到df变量中
# df = pd.read_csv('your_data.csv')

# 定义需要添加后缀的列
suffix_columns = ['dep_time', 'cap', 'duration', 'pax', 'unit_price', 'competitor_price', 'hour', 'minute']

# 定义不需要改变的列
keep_columns = ['flt_no', 'aircraft', 'a', 'b', 'c', 'year', 'month', 'day', 'weekday']

# 定义要删除的列
drop_columns = ['legs', 'leg_no', 'from', 'to']

# 定义leg_no到后缀的映射
leg_suffix_map = {
    1: 'ab',
    2: 'bc',
    3: 'ac'
}

# 创建存储结果和异常组的列表
result = []
abnormal_groups = []
skipped_groups = 0

# 首先获取分组信息，以便确定总数用于进度条
groups = df.groupby(['year', 'month', 'day', 'flt_no'])
total_groups = len(groups)
print(f"总共需要处理 {total_groups} 个航班组")

# 使用tqdm创建进度条
for (year, month, day, flt_no), group in tqdm(groups, total=total_groups, desc="处理航班组"):
    # 检查组是否包含3个航段
    if len(group) != 3:
        abnormal_info = {
            'year': year,
            'month': month,
            'day': day,
            'flt_no': flt_no,
            'row_count': len(group),
            'leg_numbers': group['leg_no'].tolist()
        }
        abnormal_groups.append(abnormal_info)
        skipped_groups += 1
        continue
    
    # 检查leg_no是否为1,2,3
    leg_nos = sorted(group['leg_no'].tolist())
    if leg_nos != [1, 2, 3]:
        abnormal_info = {
            'year': year,
            'month': month,
            'day': day,
            'flt_no': flt_no,
            'row_count': len(group),
            'leg_numbers': group['leg_no'].tolist()
        }
        abnormal_groups.append(abnormal_info)
        skipped_groups += 1
        continue
        
    new_row = {}
    
    # 处理不变的列
    for col in keep_columns:
        new_row[col] = group[col].iloc[0]
    
    # 处理需要添加后缀的列
    for col in suffix_columns:
        if col in group.columns:  # 确保列存在
            sorted_group = group.sort_values('leg_no')
            for _, row in sorted_group.iterrows():
                leg_no = row['leg_no']
                suffix = leg_suffix_map[leg_no]  # 使用映射获取对应的后缀
                new_row[f'{col}_{suffix}'] = row[col]
    
    result.append(new_row)

# 转换结果为DataFrame
print("创建结果DataFrame...")
result_df = pd.DataFrame(result)
abnormal_df = pd.DataFrame(abnormal_groups)

# 打印统计信息
print(f"\n总分组数：{skipped_groups + len(result)}")
print(f"被跳过的分组数：{skipped_groups}")
print(f"保留的分组数：{len(result)}")

# 显示异常组的情况
print("\n异常组情况：")
print(abnormal_df.head())

# 显示结果的前几行
print("\n合并后数据形状：", result_df.shape)
print("\n前5行数据：")
pd.set_option('display.max_columns', None)  # 显示所有列
print(result_df.head())

# 保存结果
print("保存结果到CSV文件...")
result_df.to_csv('./merged_flights.csv', index=False)
print("处理完成！")

总共需要处理 43511 个航班组


处理航班组: 100%|████████████████████████| 43511/43511 [02:12<00:00, 327.92it/s]


创建结果DataFrame...

总分组数：43511
被跳过的分组数：20060
保留的分组数：23451

异常组情况：
   year  month  day flt_no  row_count leg_numbers
0  2024      7    1   2571          2      [1, 2]
1  2024      7    1   2591          2      [2, 1]
2  2024      7    1   2592          2      [1, 2]
3  2024      7    1   3575          2      [2, 1]
4  2024      7    1   3576          2      [1, 2]

合并后数据形状： (23451, 30)

前5行数据：
  flt_no aircraft    a    b    c  year  month  day  weekday  cap_ab  cap_bc  \
0   2519      738  URC  LHW  CSX  2024      7    1        0     172     172   
1   2520      738  CSX  LHW  URC  2024      7    1        0     172     172   
2   2533      738  URC  UYN  FOC  2024      7    1        0     166     166   
3   2534      738  FOC  UYN  URC  2024      7    1        0     166     166   
4   2553      738  URC  NAO  WNZ  2024      7    1        0     172     172   

   cap_ac  duration_ab  duration_bc  duration_ac  pax_ab  pax_bc  pax_ac  \
0       0         2.72         2.53         5.25     11

In [17]:
result_df.to_csv('./merged_flights.csv', index=False)